In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import sqlite3
from rubin_sim.data import get_baseline
import matplotlib.pyplot as plt
%matplotlib inline
import healpy as hp

In [3]:
# or replace with any local file you want to use
baseline_db = get_baseline()
print(baseline_db)

/home/bregeon/Rubin/FINK/rubin_sim/rubin_sim_data/sim_baseline/baseline_v2.0_10yrs.db


In [4]:
conn = sqlite3.connect(baseline_db)

In [ ]:
# In the near future, 'summaryallprops' will be replaced with 'observations'
df = pd.read_sql('select * from observations;', conn)

In [ ]:
# Make any changes you want to the dataframe
df

In [ ]:
conn.close()

In [ ]:
import numpy as np
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord
from astropy import units as u

In [ ]:
# Take the time and ra/dec of GW170817 in the future as GRB270817
print('One GRB')
grb_datetime = '2027-08-17T12:41:04.4'
grb_time = Time(grb_datetime, format='isot', scale='utc')
print(f'mjd = {grb_time.mjd} for date_time = {grb_time.fits}')
#grb_ra = "19h09m48.08s"   # ra= 13h 09m 48.08s   J2000
#grb_dec = "-23d22m53.3s"  # dec= −23° 22′ 53.3″  J2000
grb_ra = "0h0m0.0s"   # ra= 13h 09m 48.08s   J2000
grb_dec = "+0d00m00.0s"  # dec= −23° 22′ 53.3″  J2000
grb_coord = SkyCoord(grb_ra, grb_dec, frame='icrs')
print(f'GRB coordinates {grb_coord}')

In [ ]:
# Observe one week before and one month after, need 4 weeks
one_week = TimeDelta(7, format='jd')
four_weeks = TimeDelta(28, format='jd')
obs_start = grb_time - one_week
obs_end = grb_time + four_weeks
print(f'Observations Start {obs_start} / End {obs_end}')
print(f'Observations Start {obs_start.mjd} / End {obs_end.mjd}')

In [ ]:
# get time span
df_time = df[df['observationStartMJD']>obs_start.mjd][df['observationStartMJD']<obs_end.mjd]
df_time

In [ ]:
# Angular separation with SkyCoord.separation
# Rubin FOV is 47 square degree for a 3.5-degree diameter, hence 1.7 deg separation radius.
df_time['Separation'] = SkyCoord(df_time['fieldRA'], df_time['fieldDec'], unit="deg").separation(grb_coord).degree
df_time.head()

In [ ]:
plt.rcParams["figure.figsize"] = [15, 6]
m_fig, m_ax = plt.subplots(1, 2)
m_ax.flatten()
_fm = m_ax[0].hist2d(df['fieldRA'], df['fieldDec'], cmap=plt.cm.jet)
m_ax[0].set_title('Full Map Ra/Dec')
_om = m_ax[1].hist2d(df_time['fieldRA'], df_time['fieldDec'], cmap=plt.cm.jet)
m_ax[1].set_title('One Month Map Ra/Dec')


In [ ]:
df_sky = df_time[df_time['Separation']<1.7]
df_sky

In [ ]:
df_sky.columns

In [ ]:
plt.rcParams["figure.figsize"] = [15, 12]
fig, ax = plt.subplots(2, 2)
_filter = ax[0][0].hist(df_sky['filter'])
ax[0][0].set_title('Filters')
_sep = ax[0][1].hist(df_sky['Separation'])
ax[0][1].set_title('Separation')
_mjd = ax[1][0].hist(df_sky['observationStartMJD'])
ax[1][0].set_title('Obs Start MJD')
_depth = ax[1][1].hist(df_sky['fiveSigmaDepth'])
ax[1][1].set_title('5 Sigma Depth')


In [ ]:
plt.scatter(df_sky['observationStartMJD'], df_sky['fiveSigmaDepth'])
plt.gca().invert_yaxis()
plt.title('5 Sigma depth')

In [ ]:
# Import the primary photometry classes from rubin_sim.photUtils
import os
import rubin_sim.photUtils.Bandpass as Bandpass
import rubin_sim.photUtils.Sed as Sed

In [ ]:
fdir = '/home/bregeon/Rubin/FINK/rubin_sim/rubin_sim_data'
fdir = os.path.join(fdir, 'throughputs', 'baseline')

# Read the throughput curves
filterlist = ['u', 'g', 'r', 'i', 'z', 'y']
filtercolors = {'u':'b', 'g':'c', 'r':'g', 'i':'orange', 'z':'r', 'y':'m'}

lsst = {}
for f in filterlist:
    lsst[f] = Bandpass()
    lsst[f].readThroughput(os.path.join(fdir, f'total_{f}.dat'))

In [ ]:
# compute observation time in the GRB time frame, i.e. from GRB T_0
obs_times_grb_frame = df_sky['observationStartMJD'] - grb_time.mjd
time_bins = obs_times_grb_frame[obs_times_grb_frame>0]
time_bins.to_dict()

In [ ]:
# compute spectra at observation time bins
import afterglowpy as grb
from orphans.grb_interface import make_grb_spectrum, dump_wl_Fnu_spectrum

Fnu_Jy = dict()
for obs_id, obs_t in time_bins.to_dict().items():
    wl_full_band, freq_full_band, t, Fnu_Jy[obs_id] = make_grb_spectrum(E0=1.0e54, thetaObs=0.1, thetaCore=0.05, t=obs_t * grb.day2sec)


In [ ]:
Fnu_Jy

In [ ]:
def compute_mags(i, wls, fnus, obs_t):
    new_grb_sed = Sed()
    new_grb_sed.wavelen = np.array(wls)
    new_grb_sed.fnu = np.array(fnus)
    # convert fnu to flambda
    new_grb_sed.fnuToflambda()
    # Calculate expected AB magnitudes. 
    new_grb_mags = {}
    for f in filterlist:
        new_grb_mags[f] = new_grb_sed.calcMag(lsst[f])
    # time is one column
    new_grb_mags['obs_time'] = obs_t
    # Make a dataframe just to get a nice output cell.
    return pd.DataFrame(new_grb_mags, index=[i])

In [ ]:
# compute magnitudes
obs_list = list()
for obs_id, fnu_val in Fnu_Jy.items():
    obs_t = time_bins[obs_id]
    df = compute_mags(obs_id, wl_full_band, fnu_val, obs_t)
    obs_list.append(df)
obs_df = pd.concat(obs_list)
obs_df

In [ ]:
# keep only "real" observation for the right filter
x_times = list()
y_mags = list()
z_colors = list()
mags_lim = list()
for obs_id, obs in obs_df.iterrows():
    # obs_id = one[0]
    #obs_time = one[1]['obs_time']
    #print(obs_id, obs_time)
    filt = df_sky[df_sky['observationId']==obs_id]['filter']
    lim = df_sky[df_sky['observationId']==obs_id]['fiveSigmaDepth']
    print(obs_id, obs['obs_time'], filt.values[0], obs[filt].values[0], lim.values[0])
    x_times.append(obs['obs_time'] + grb_time.mjd)
    y_mags.append(obs[filt].values[0])
    z_colors.append(filtercolors[filt.values[0]])
    mags_lim.append(lim.values[0])

In [ ]:
from matplotlib.lines import Line2D
# plot pseudo observed light curve
for x, y, z, m in zip(x_times, y_mags, z_colors, mags_lim):
    plt.scatter(x, y, c=z)
    plt.scatter(x, m, c=z, marker='2')
plt.gca().invert_yaxis()
plt.title('Pseudo observed light curve (all points)')
plt.xlabel('Time in MJD')
plt.ylabel('Observed magnitdue')
legend_elements = list()
for filt in filterlist:
    fcolor = filtercolors[filt]
    legend_elements.append(Line2D([0], [0], marker='o', color=fcolor, label=filt,
                                  markerfacecolor=fcolor, markersize=10))
plt.legend(handles=legend_elements)
plt.show()
print(grb_time.mjd)

In [ ]:
# plot pseudo observed light curve
for x, y, z, m in zip(x_times, y_mags, z_colors, mags_lim):
    if y<m:
        plt.scatter(x, y, c=z)
        plt.scatter(x, m, c=z, marker='2')
plt.gca().invert_yaxis()
plt.title('Pseudo observed light curve (life is hard!)')
plt.xlabel('Time in MJD')
plt.ylabel('Observed magnitdue')
legend_elements_2 = list()
for filt in filterlist:
    fcolor = filtercolors[filt]
    legend_elements_2.append(Line2D([0], [0], marker='o', color=fcolor, label=filt,
                                  markerfacecolor=fcolor, markersize=10))
plt.legend(handles=legend_elements_2)
plt.show()

In [ ]:
from orphans.grb_interface import make_grb_spectrum, make_grb_light_curve, dump_wl_Fnu_spectrum
from orphans.tools import flux_to_mag
full_nu, full_t, full_Fnu_Jy = make_grb_light_curve(E0=1.0e54, thetaObs=0.1, thetaCore=0.05)

In [ ]:
def ObsTime(t, mag):
    index = []
    for i in range(len(mag)-1):
        if mag[i] < 24.5:
            index.append(i)
    if not index:
        return 0
    else:
        dt = (t[max(index)] - t[min(index)])*grb.sec2day
        return dt

In [ ]:
#plt.plot(full_t/86400.+grb_time.mjd, flux_to_mag(full_Fnu_Jy))
plt.plot(full_t/86400., flux_to_mag(full_Fnu_Jy))
plt.xlabel('Time (days)')
plt.ylabel('Magnitude')
plt.gca().invert_yaxis()
plt.xscale('log')
plt.axhline(y=24.5, color='black', linestyle=':', label='LSST limiting magnitude')
plt.xlim(0.01,1000)
#plt.xlim(61640,61650)
plt.ylim(27,16)
plt.legend()
plt.text(20, 18, f'Observability = {ObsTime(full_t, flux_to_mag(full_Fnu_Jy)):.1f} days', size=18)
plt.show()

